In [256]:
import gpt as g
import numpy as np

In [275]:
grid = g.grid([128,2,2,4], g.double)

In [276]:
U = g.qcd.gauge.unit(grid)

In [294]:
m0 = 0.02
M5 = 1.6
w0 = 1 - M5
mob = g.qcd.fermion.mobius(U, mass=m0, Ls=64, b=1, c=0, M5=M5, 
                           boundary_phases=[1,1,1,1])

In [295]:
pc = g.qcd.fermion.preconditioner
inv = g.algorithms.inverter
exact_prec = 1e-8
cg_e_inner = inv.cg({"eps": 1e-4, "eps_abs": exact_prec * 0.3, "maxiter": 40000, "miniter": 50, "fail_if_not_converged": True})
cg_e_inner = inv.preconditioned(pc.eo2_ne(), cg_e_inner)
slv = inv.defect_correcting(
    inv.mixed_precision(cg_e_inner, g.single, g.double),
    eps=exact_prec,
    maxiter=100,
)

In [296]:
prop4d = mob.propagator(slv)

In [297]:
p0 = 2*np.pi/128
print(p0)
src = g.mspincolor(grid)
src @= g.identity(src)

0.04908738521234052


In [298]:
prop1 = g.sum(g.exp_ixp(p=np.array([-p0,0,0,0])) * prop4d * g.exp_ixp(p=np.array([p0,0,0,0])) * src) / grid.gsites

GPT :    2398.314753 s : cg: converged in 64 iterations
GPT :    2398.672853 s : cg: converged in 64 iterations
GPT :    2399.011956 s : cg: converged in 64 iterations
GPT :    2399.349175 s : cg: converged in 64 iterations
GPT :    2399.687662 s : cg: converged in 64 iterations
GPT :    2400.024377 s : cg: converged in 64 iterations
GPT :    2400.363743 s : cg: converged in 64 iterations
GPT :    2400.700684 s : cg: converged in 64 iterations
GPT :    2401.034713 s : cg: converged in 64 iterations
GPT :    2401.371787 s : cg: converged in 64 iterations
GPT :    2401.706419 s : cg: converged in 64 iterations
GPT :    2402.046149 s : cg: converged in 64 iterations
GPT :    2402.842997 s : cg: converged in 99 iterations
GPT :    2403.382697 s : cg: converged in 99 iterations
GPT :    2403.913914 s : cg: converged in 99 iterations
GPT :    2404.453006 s : cg: converged in 99 iterations
GPT :    2404.987738 s : cg: converged in 99 iterations
GPT :    2405.520377 s : cg: converged in 99 ite

In [299]:
Zq = 1-w0**2
g.trace(prop1)/12/m0*(np.sin(p0)**2/Zq**2 + m0**2)

(1.0081530436246886+5.396454324290851e-11j)

In [300]:
g.trace(prop1 * g.gamma[0])/12/(-1j*np.sin(p0)/Zq)*(np.sin(p0)**2/Zq**2 + m0**2)

(1.0037896717203545+4.232799672824428e-10j)

In [ ]:
# verified conventions of (III.1) of https://arxiv.org/pdf/hep-lat/9810020